<a href="https://colab.research.google.com/github/ladybug-tentoumushi/smolvla-libero-spatial/blob/main/Physical_AI_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

参考URL：
https://qiita.com/hEnka/items/cc5fd872eb0bf7cd3abc

# 環　境

In [ ]:
import sys, subprocess, json, os, shutil
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from uuid import uuid4

assert (3, 10) <= sys.version_info[:2] < (3, 14), "GenesisにはPython 3.10〜3.13が必要です。"
subprocess.run([sys.executable, "-c", "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'ColabをGPUランタイムに変更してください'; print('GPU:', torch.cuda.get_device_name(0))"], check=True)
subprocess.run(["nvidia-smi"], check=True)
print("Python:", sys.version)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP_ROOT = Path("/content/drive/MyDrive/PysicalAI/GO2")
print("保存先:", BACKUP_ROOT)

# Step 2の保存先だけ分ける
STEP1_ROOT = BACKUP_ROOT
BACKUP_ROOT = STEP1_ROOT / 'step2'
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
print('Step 2保存先:', BACKUP_ROOT)

# Genesisをgit clone

In [ ]:
GENESIS_COMMIT = "02908fe6474d8b073b662347a8794614fa250658"
REPO = Path("/content/genesis-go2-" + GENESIS_COMMIT[:12])
if not REPO.exists():
  REPO.mkdir()
  subprocess.run(["git", "init", str(REPO)], check=True)
  subprocess.run(["git", "-C", str(REPO), "remote", "add", "origin", "https://github.com/Genesis-Embodied-AI/Genesis.git"], check=True)
actual = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True)
if actual.returncode != 0:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GENESIS_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", GENESIS_COMMIT], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_sha == GENESIS_COMMIT, "版が違います。既存フォルダを確認してください。"
print("Genesis commit:", actual_sha)
print("作業場所:", REPO)


# 環境を固定

In [ ]:
from importlib import metadata
from packaging.version import Version
assert Version(metadata.version("torch").split("+")[0]) >= Version("2.6.0"), "PyTorchが古いです。新しいColabランタイムを使ってください。"
constraints = []
for package in ["torch", "torchvision", "torchaudio"]:
    try:
        constraints.append(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        pass
constraints_path = REPO / "colab-torch-constraints.txt"
constraints_path.write_text("\n".join(constraints) + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints_path),
                "-e", str(REPO), "rsl-rl-lib==5.0.0", "tensorboard"], check=True)
print("インストール完了。次のセルで実際の読み込みを確認します。")

# バージョンの記録

In [ ]:
subprocess.run([sys.executable, "-c", "import torch, genesis, torchvision; from rsl_rl.runners import OnPolicyRunner; from tensordict import TensorDict; assert torch.cuda.is_available(); print('import OK')"], cwd=REPO, check=True)
versions = {
    "genesis_commit": GENESIS_COMMIT,
    "python": sys.version,
    "packages": {p: metadata.version(p) for p in ["genesis-world", "rsl-rl-lib", "torch", "torchvision", "tensordict", "tensorboard"]},
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True).strip(),
}
(REPO / "versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
(REPO / "pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print(json.dumps(versions, indent=2))

# Colab用の補助ファイルを作成  
公式の環境・PPO設定を読み込み、①既存結果の削除防止、②25回ごとのDrive保存、③追加学習、④動画評価を加えます。 公式 go2_env.py は変更せず、録画用のコピーにだけ、シーン構築前のカメラ追加を行います。 公式 go2_train.py のmain処理は使わないため、同名ログの自動削除も実行されません。  
「Unitree Go2」に強化学習で歩行を習得させ、学習結果を保存し、その性能を数値や動画で確認

## ①元のコードを確認

In [ ]:
import hashlib  # データからハッシュ値を計算するPythonライブラリ

expected_hashes = {'go2_env.py': '22a08b9f1f5c765732a7b3d04c98b3cb6a25b565aa024479a0526bbc654c1a63', 'go2_train.py': '9d78b33a90e8833be648c302a818f1b10ebfb632d0f5de5214f68bf1d714cb00', 'go2_eval.py': '2e80b1c65dea21b28a97b2c11eb6f035bfa7ded87701e3e2abe8e9bd41824bd0'}
for name, expected in expected_hashes.items():
    path = REPO / "examples" / "locomotion" / name
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"公式ファイルが想定と異なります: {name}"

## 学習用ファイルを作成

In [ ]:

(REPO / 'colab_train.py').write_text(r'''"""Colab helper: use the pinned official Go2 environment/configuration."""
import argparse
import json
import os
import pickle
import shutil
import time
import sys
from pathlib import Path

import genesis as gs
from rsl_rl.runners import OnPolicyRunner

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "examples" / "locomotion"))
from go2_env import Go2Env
from go2_train import get_cfgs, get_train_cfg


def copy_complete(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)

# 学習条件を指定する
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--num-envs", type=int, default=1024)  # 仮想環境を1024個並列実行
    p.add_argument("--iterations", type=int, default=401) # 学習の繰り返し回数：401回
    p.add_argument("--backup-root", required=True)
    p.add_argument("--resume", default="")
    p.add_argument("--seed", type=int, default=1)
    p.add_argument("--save-interval", type=int, default=25) # 25反復ごとに保存
    args = p.parse_args()
    if Path(args.name).name != args.name or args.name in (".", ".."):
        raise ValueError("Use a simple experiment name, not a path.")
    if args.iterations < 1 or args.num_envs < 4 or args.save_interval < 1 or args.seed < 0:
        raise ValueError("iterations/save-interval >= 1, num-envs >= 4, seed >= 0 are required.")
    log = ROOT / "logs" / args.name
    backup = Path(args.backup_root) / args.name
    if log.exists() or backup.exists():
        raise FileExistsError("Experiment already exists. Choose a NEW name; nothing was deleted.")

# 学習条件を読み込む
    if args.resume:
        checkpoint = Path(args.resume).resolve()
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        with (checkpoint.parent / "cfgs.pkl").open("rb") as f:
            cfgs = pickle.load(f)
    else:
        cfgs = [*get_cfgs(), get_train_cfg(args.name)]
    env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = cfgs # env_cfg: 環境設定、 obs_cfg: 観測設定、reward_cfg: 報酬設定、command_cfg: コマンド設定、train_cfg: 学習設定

    # Step 2で追加
    command_cfg["lin_vel_x_range"] = [0.2, 0.8] # 前進速度のみ0.2-0.8に変更
    command_cfg["lin_vel_y_range"] = [0.0, 0.0]
    command_cfg["ang_vel_range"] = [0.0, 0.0]

    train_cfg["run_name"] = args.name
    train_cfg["save_interval"] = args.save_interval
    train_cfg["logger"] = "tensorboard"
    log.mkdir(parents=True)
    backup.mkdir(parents=True)
    # Save BEFORE Go2Env scales the reward weights in place.
    with (log / "cfgs.pkl").open("wb") as f:
        pickle.dump(cfgs, f)
    (log / "cfgs.json").write_text(json.dumps(dict(zip(
        ["environment", "observations", "rewards", "commands", "training"], cfgs)), indent=2), encoding="utf-8")
    (log / "run.json").write_text(json.dumps(vars(args), indent=2), encoding="utf-8")
    sources = log / "sources"
    sources.mkdir()
    for name in ("go2_env.py", "go2_train.py", "go2_eval.py"):
        shutil.copy2(ROOT / "examples" / "locomotion" / name, sources / name)
    for name in ("colab_train.py", "colab_eval.py", "go2_env_video.py"):
        shutil.copy2(ROOT / name, sources / name)
    for name in ("versions.json", "pip-freeze.txt"):
        shutil.copy2(ROOT / name, log / name)
    shutil.copytree(log, backup, dirs_exist_ok=True)

# 学習途中のデータを保存
    class BackedUpRunner(OnPolicyRunner):
        def save(self, path, infos=None):
            super().save(path, infos)
            # Called only AFTER torch.save has completed; no half-written checkpoint is copied.
            copy_complete(path, backup / Path(path).name)
            # Keep TensorBoard logs available even when the runtime disconnects.
            for event in log.glob("events.out.tfevents.*"):
                copy_complete(event, backup / event.name)
            print("Drive checkpoint:", backup / Path(path).name, flush=True)

# 強化学習を実行
    gs.init(backend=gs.gpu, precision="32", logging_level="warning",
            seed=args.seed, performance_mode=True)
    env = Go2Env(args.num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg)
    runner = BackedUpRunner(env, train_cfg, str(log), device=gs.device)
    if args.resume:
        runner.load(str(checkpoint), map_location=gs.device)
        # rsl-rl 5.0.0 stores the index of the last COMPLETED iteration.
        runner.current_learning_iteration += 1
    started = time.perf_counter()
    runner.learn(num_learning_iterations=args.iterations, init_at_random_ep_len=True)
    (log / "training_summary.json").write_text(json.dumps({
        "requested_iterations": args.iterations,
        "environment_steps_this_run": args.num_envs * args.iterations * train_cfg["num_steps_per_env"],
        "elapsed_seconds": time.perf_counter() - started,
        "seed": args.seed,
        "resume_checkpoint": args.resume or None,
    }, indent=2), encoding="utf-8")
    shutil.copytree(log, backup, dirs_exist_ok=True)
    print("Training finished. Backup:", backup, flush=True)


if __name__ == "__main__":
    main()
''', encoding='utf-8')

(REPO / 'colab_eval.py').write_text(r'''"""Finite, repeatable evaluation with per-rollout metrics and step-level CSV."""
import argparse
import csv
import hashlib
import json
import math
import os
import pickle
import random
import statistics
import sys
import time
from pathlib import Path

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "examples" / "locomotion"))


def summarize_rollout(seed, samples, dt, max_error):
    valid = [s for s in samples if not s["terminated"]]
    errors = [abs(s["forward_speed_m_s"] - s["command_m_s"]) for s in valid]
    resets = sum(s["terminated"] for s in samples)
    mae = statistics.mean(errors) if errors else None
    return {
        "seed": seed, "simulated_seconds": len(samples) * dt,
        "mean_forward_speed_m_s": statistics.mean(s["forward_speed_m_s"] for s in valid) if valid else None,
        "mean_abs_forward_speed_error_m_s": mae,
        "non_timeout_terminations": resets,
        "valid_velocity_samples": len(valid),
        "success": bool(valid) and resets == 0 and mae <= max_error,
    }


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--seconds", type=float, default=10.0)
    p.add_argument("--episodes", type=int, default=5)
    p.add_argument("--seed", type=int, default=1000)
    p.add_argument("--max-speed-error", type=float, default=0.15)
    p.add_argument("--video", action="store_true")
    args = p.parse_args()
    if not math.isfinite(args.seconds) or args.seconds < 0.02:
        p.error("seconds must be finite and >= 0.02")
    if not math.isfinite(args.max_speed_error) or args.max_speed_error < 0:
        p.error("max-speed-error must be finite and >= 0")
    if args.episodes < 1 or not 0 <= args.seed <= 2**32 - args.episodes:
        p.error("episodes must be positive and seeds must fit uint32")
    return args

# 動画を生成
def main():
    args = parse_args()
    import numpy as np
    import torch
    import genesis as gs
    from rsl_rl.runners import OnPolicyRunner

    checkpoint = Path(args.checkpoint).resolve()
    output = Path(args.output).resolve()
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    # Never mix a failed rerun with earlier successful metrics/video.
    output.mkdir(parents=True, exist_ok=False)
    with (checkpoint.parent / "cfgs.pkl").open("rb") as f:
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(f)
    reward_cfg["reward_scales"] = {}
    env_cfg["episode_length_s"] = max(env_cfg["episode_length_s"], args.seconds + 1)
    if args.video:
        from go2_env_video import Go2Env
    else:
        from go2_env import Go2Env
    gs.init(backend=gs.gpu, precision="32", logging_level="warning", seed=args.seed)
    env = Go2Env(1, env_cfg, obs_cfg, reward_cfg, command_cfg, show_viewer=False)
    runner = OnPolicyRunner(env, train_cfg, log_dir=None, device=gs.device)
    runner.load(str(checkpoint), map_location=gs.device)
    policy = runner.get_inference_policy(device=gs.device)
    frames = max(1, round(args.seconds / env.dt))
    results = []
    started = time.perf_counter()
    fields = ["episode", "seed", "step", "time_s", "command_m_s", "forward_speed_m_s", "terminated"]
    with (output / "trajectory.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for episode in range(args.episodes):
            seed = args.seed + episode
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
            obs = env.reset()
            samples = []
            recording = args.video and episode == 0
            if recording:
                env.camera.start_recording(save_to_filename=str(output / "go2.mp4"), fps=25)
            try:
                with torch.inference_mode():
                  # step2 速度変化
                    for step in range(frames):
                        # 時間の経過で速度を切替
                        t = round(step * env.dt, 8)
                        if t < 10.0:
                          command = 0.2
                        elif t < 20.0:
                          command = 0.4
                        else:
                          command = 0.8

                        # 横移動と旋回はゼロ
                        env.base_lin_vel[0, 0] = command
                        env.base_ang_vel[0, 0] = 0.0

                        # 観測の更新
                        env._update_observation()
                        obs = env.get_observations()

                        # カメラの位置を更新
                        if recording:
                          env.camera.set_pose(
                            pos=(x + 1.2, y - 1.8, z + 0.8),
                            lookat=(x, y, z),
                        )
                        action = policy(obs)
                        if not torch.isfinite(action).all().item():
                            raise FloatingPointError("Policy produced non-finite actions")
                        obs, _, done, _ = env.step(action)
                        terminated = bool(done[0])
                        # Terminal-step velocities have already been reset by Go2Env.
                        speed = None if terminated else env.base_lin_vel[0, 0].item()
                        if speed is not None and not math.isfinite(speed):
                            raise FloatingPointError("Simulator produced non-finite velocity")
                        row = dict(episode=episode + 1, seed=seed, step=step + 1,
                                   time_s=(step + 1) * env.dt, command_m_s=command,
                                   forward_speed_m_s=speed, terminated=terminated)
                        samples.append(row)
                        writer.writerow(row)
            finally:
                if recording:
                    env.camera.stop_recording()
                stream.flush()
            results.append(summarize_rollout(seed, samples, env.dt, args.max_speed_error))
            print(json.dumps(results[-1]), flush=True)
    errors = [r["mean_abs_forward_speed_error_m_s"] for r in results
              if r["mean_abs_forward_speed_error_m_s"] is not None]
    metrics = {
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
        "evaluation": vars(args), "command_config": command_cfg,
        "success_definition": "No termination and forward-speed MAE <= max_speed_error over the entire rollout",
        "success_rate": statistics.mean(r["success"] for r in results),
        "mean_abs_forward_speed_error_m_s": statistics.mean(errors) if errors else None,
        "std_abs_forward_speed_error_m_s": statistics.stdev(errors) if len(errors) > 1 else None,
        "episodes_with_valid_velocity": len(errors),
        "non_timeout_terminations": sum(r["non_timeout_terminations"] for r in results),
        "elapsed_seconds": time.perf_counter() - started,
        "episode_results": results,
        "video_episode": 1 if args.video else None,
        "note": "Fixed initial pose and fixed commands can give identical trials despite different seeds. "
                "This measures this simulated condition only, not robustness or real-world success. "
                "Reset-step velocities are excluded; terminations include tilt and solver errors.",
    }
    name = "metrics-video.json" if args.video else "metrics.json"
    (output / name).write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()
''', encoding='utf-8')

original = (REPO / "examples" / "locomotion" / "go2_env.py").read_text(encoding="utf-8")
anchor = "        self.scene.build(n_envs=num_envs)"
assert original.count(anchor) == 1, "カメラ挿入位置が一致しません。版を確認してください。"
camera = "        self.camera = self.scene.add_camera(res=(640, 480), pos=(2.5, -3.0, 2.0), lookat=(0.0, 0.0, 0.4), fov=40, GUI=False, env_idx=0)\n"
(REPO / "go2_env_video.py").write_text(original.replace(anchor, camera + anchor), encoding="utf-8")
for name in ["colab_train.py", "colab_eval.py", "go2_env_video.py"]:
    compile((REPO / name).read_text(encoding="utf-8"), name, "exec")
print("補助ファイル3本を作成しました。")



# 動作テスト（256環境×2回）  
少数の仮想Go2で、物理シミュレーション→PPO→保存までつながるか確認します。 初回はGPU向けのコンパイルがあり、ログの進みが遅くても異常とは限りません。 Smoke test OK が出れば次へ。2回だけなので、まだ歩けなくて正常です。  


In [ ]:
# 生成済みの学習スクリプトに、明示的な終了処理を追加する
path = REPO / "colab_train.py"
source = path.read_text(encoding="utf-8")

before = '''    shutil.copytree(log, backup, dirs_exist_ok=True)
    print("Training finished. Backup:", backup, flush=True)'''

after = '''    # TensorBoardの未書き込みログを保存して閉じる
    if runner.logger.writer is not None:
        runner.logger.writer.flush()
        runner.logger.writer.close()

    # ログを閉じた後に、最終版をDriveへコピー
    shutil.copytree(log, backup, dirs_exist_ok=True)
    print("Training and backup finished.", flush=True)

    # Pythonの自動終了に任せず、ここでGenesisを終了
    print("Genesis cleanup starting...", flush=True)
    gs.destroy()
    print("Genesis cleanup finished.", flush=True)

    print("Training finished. Backup:", backup, flush=True)'''

if after in source:
    print("終了処理はすでに追加済みです。")
else:
    assert source.count(before) == 1, (
        "置換対象が一致しません。colab_train.pyの内容を確認してください。"
    )
    updated = source.replace(before, after)
    compile(updated, str(path), "exec")
    path.write_text(updated, encoding="utf-8")
    print("終了処理を追加しました。")

In [ ]:
path = REPO / "colab_train.py"
source = path.read_text(encoding="utf-8")

before = '''    print("Genesis cleanup starting...", flush=True)
    gs.destroy()
    print("Genesis cleanup finished.", flush=True)

    print("Training finished. Backup:", backup, flush=True)'''

after = '''    # 学習・ログのクローズ・Driveへのコピーが済んだ後だけ実行する
    import hashlib
    import torch

    final_name = f"model_{runner.current_learning_iteration}.pt"
    local_model = log / final_name
    drive_model = backup / final_name

    def sha256_file(path):
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    # 最後のモデルが存在することを確認
    for file in (local_model, drive_model):
        if not file.is_file() or file.stat().st_size == 0:
            raise RuntimeError(f"保存されたモデルを確認できません: {file}")

    # Colab内とDrive上のファイル内容が一致することを確認
    if sha256_file(local_model) != sha256_file(drive_model):
        raise RuntimeError("モデルのコピー内容が一致しません。")

    # Drive上のモデルを実際に読み込めることを確認
    saved = torch.load(drive_model, map_location="cpu", weights_only=True)
    if not isinstance(saved, dict) or not saved:
        raise RuntimeError("モデルの保存内容が想定と異なります。")
    del saved

    print("Checkpoint verified:", drive_model, flush=True)
    print("Training finished. Backup:", backup, flush=True)
    print("EGL終了処理を省略して、学習用プロセスを終了します。", flush=True)

    sys.stdout.flush()
    sys.stderr.flush()
    os._exit(0)'''

if after in source:
    print("回避策はすでに適用済みです。")
else:
    assert source.count(before) == 1, (
        "置換対象が一致しません。前回の終了処理追加セルを確認してください。"
    )
    updated = source.replace(before, after)
    compile(updated, str(path), "exec")
    path.write_text(updated, encoding="utf-8")
    print("保存確認後に終了する回避策を適用しました。")

**Step 2変更②：Step 1のモデルを選択します。**
候補が1個なら自動選択します。複数ならSTEP1_PATHに表示された絶対パスを指定してください。
信頼できる自分のモデルのみ読み込んでください。設定とモデルが読み込めれば正常です。

In [ ]:
import pickle, torch
STEP1_PATH = ''
candidates = sorted(p for p in STEP1_ROOT.rglob('model_400.pt')
                    if BACKUP_ROOT not in p.parents and (p.parent / 'cfgs.pkl').is_file())
print('候補:', *candidates, sep='\n')
if not STEP1_PATH:
    assert len(candidates) == 1, 'STEP1_PATHに使用するmodel_400.ptを指定してください。'
    STEP1_PATH = str(candidates[0])
STEP1_CHECKPOINT = Path(STEP1_PATH)
with (STEP1_CHECKPOINT.parent / 'cfgs.pkl').open('rb') as f:
    saved_cfgs = pickle.load(f)
assert saved_cfgs[0]['num_actions'] == 12
assert saved_cfgs[3]['num_commands'] == 3
saved_model = torch.load(STEP1_CHECKPOINT, map_location='cpu', weights_only=False)
assert saved_model['iter'] == 400
del saved_model
print('再利用するモデル:', STEP1_CHECKPOINT)
print('元の速度設定:', saved_cfgs[3])


In [ ]:
import sys
import subprocess
from datetime import datetime
from zoneinfo import ZoneInfo
from uuid import uuid4

def new_name(prefix):
    return (
        prefix + "-"
        + datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d-%H%M%S")
        + "-" + uuid4().hex[:6]
    )

def run_training(name, n_envs, iterations,
                 resume="", seed=1, save_interval=25):
    cmd = [
        sys.executable,
        "-X", "faulthandler",  # 異常終了した場所の情報を出す
        "-u",
        str(REPO / "colab_train.py"),
        "--name", name,
        "--num-envs", str(n_envs),
        "--iterations", str(iterations),
        "--backup-root", str(BACKUP_ROOT),
        "--seed", str(seed),
        "--save-interval", str(save_interval),
    ]
    if resume:
        cmd += ["--resume", str(resume)]

    # Driveとは別に、Colab内へ診断ログを残す
    (BACKUP_ROOT / "diagnostics").mkdir(exist_ok=True)
    log_path = BACKUP_ROOT / "diagnostics" / f"{name}-diagnostic.log"
    print("診断ログ:", log_path, flush=True)

    # 学習プログラムの出力を、画面とファイルの両方に出す
    with log_path.open("w", encoding="utf-8") as log_file:
        with subprocess.Popen(
            cmd,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log_file.write(line)
                log_file.flush()

            returncode = process.wait()

    if returncode != 0:
        raise RuntimeError(
            f"学習プログラムが停止しました（終了コード: {returncode}）。"
            f"上のログを確認してください。ログ保存先: {log_path}"
        )

SMOKE_NAME = new_name("step2-smoke")
run_training(SMOKE_NAME, 256, 2, resume=STEP1_CHECKPOINT)

smoke_models = list((BACKUP_ROOT / SMOKE_NAME).glob("model_*.pt"))
assert smoke_models, "Driveにモデルが見つかりません。"
print("Smoke test OK:", smoke_models)
with (BACKUP_ROOT / SMOKE_NAME / 'cfgs.pkl').open('rb') as f:
    smoke_cfgs = pickle.load(f)
assert smoke_cfgs[3]['lin_vel_x_range'] == [0.2, 0.8]
assert smoke_cfgs[1] == saved_cfgs[1], '観測設定が変わっています。'
assert smoke_cfgs[0]['num_actions'] == saved_cfgs[0]['num_actions']
SMOKE_OK = True
print('Step 1読み込み・新速度設定の保存確認OK')


# 本学習の設定  
初期設定は 1024環境×401回。公式の4096環境×101回に近い総経験量にしていますが、同じ性能になる保証はありません。 GPUメモリ不足なら NUM_ENVS = 512、経験量の目安を維持するなら ITERATIONS = 801 にします。 まず短く試したいなら101回でも構いません。学習時間はGPU・コンパイル・Driveへの保存時間で変わります。 このセルを再実行すると、新しい実験名になります。

In [ ]:
NUM_ENVS = 1024
ITERATIONS = 400
TRAIN_SEED = 1
SAVE_INTERVAL = 25
EXP_NAME = new_name("step2-variable")
LOG_DIR = REPO / "logs" / EXP_NAME
BACKUP_DIR = BACKUP_ROOT / EXP_NAME
session = {"exp_name": EXP_NAME, "num_envs": NUM_ENVS, "iterations": ITERATIONS,
           "genesis_commit": GENESIS_COMMIT, "seed": TRAIN_SEED, "save_interval": SAVE_INTERVAL}
(REPO / "active_run.json").write_text(json.dumps(session, indent=2), encoding="utf-8")
print("実験名:", EXP_NAME)
print("保存先:", BACKUP_DIR)
print("総ステップ数の目安:", NUM_ENVS * ITERATIONS * 24)

# 本学習を実行  
前進0.5 m/sを目標に練習します。 Learning iteration が進み、最後に Training finished が出れば処理完了です。 途中モデルはセル8の SAVE_INTERVAL 間隔（初期値25）でDriveへ保存します。保存前に接続が切れた分は失われます。 同じセルを再実行して既存実験エラーが出たら、新規学習はセル8から、追加学習はセル16を使います。

In [ ]:
assert globals().get("SMOKE_OK", False), "スモークテストを先に実行してください。"
run_training(EXP_NAME, NUM_ENVS, ITERATIONS, resume=STEP1_CHECKPOINT, seed=TRAIN_SEED, save_interval=SAVE_INTERVAL)
print("本学習の処理が完了しました。次にモデルと歩行を確認します。")

# 学習グラフを見る  
TensorBoardは学習の記録を見る画面です。報酬・エピソード長の推移を確認します。 ログの項目名はライブラリで変わります。グラフが上がっていても、歩行成功の判断は動画と評価を合わせて行います。 この配置では学習後に表示します。表示がブロックされても、モデルの保存成功とは別問題です。

Qiita記事との対応：記事のTensorBoard起動。

変更の種類：Colab対応。

何が変わったか・理由：ターミナルの起動コマンドを、ノートブック内で表示する%tensorboardへ変更しました。ログを見る目的は同じです。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR

# 最新モデルとDrive保存を確認  
model_100.pt のような固定名に頼らず、数字が最大のチェックポイントを選びます。 ローカルとDriveのファイル内容も比較します。モデルだけでなく cfgs.pkl と versions.json も必要です。

In [ ]:
def latest_model(folder):
    paths = list(Path(folder).glob("model_*.pt"))
    if not paths:
        raise FileNotFoundError(f"モデルがありません: {folder}")
    return max(paths, key=lambda p: int(p.stem.split("_")[-1]))

CHECKPOINT = latest_model(LOG_DIR)
drive_checkpoint = BACKUP_DIR / CHECKPOINT.name
assert drive_checkpoint.is_file()
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == hashlib.sha256(drive_checkpoint.read_bytes()).hexdigest()
for name in ["cfgs.pkl", "cfgs.json", "versions.json", "run.json", "pip-freeze.txt", "training_summary.json"]:
    assert (BACKUP_DIR / name).is_file(), name
print("最新モデル:", CHECKPOINT)
print("Driveコピー一致:", drive_checkpoint)
print("設定・バージョン・コードも保存済み:", BACKUP_DIR)

# 描画なしで複数回評価  
第2回RoboOps教材の固定seed・試行別記録をGo2へ適用します。初期値は10秒×5試行です。 EVAL_SEED から連続するseedを使い、試行別の速度・速度誤差・リセット数と集計をJSONへ保存します。 success_rate は「試行中のリセットなし、かつ平均絶対速度誤差が MAX_SPEED_ERROR 以下」の割合です。 0.15 m/sという閾値はこのノートブックで設けた暫定基準で、講義の合格基準ではありません。 リセット直後の速度は除外しますが、リセットした試行は必ず不成功になります。 初期姿勢と速度指令が固定なので、seedを変えても結果が同じ場合があります。頑健性や実機の成功率は示しません。 trajectory.csv に各ステップの指令・速度・リセットを記録します。失敗時も記録済みの行は残ります。 再実行ごとに保存先を分けます。過去モデルとの比較では秒数・seed・試行数・閾値をそろえてください。

In [ ]:
path = REPO / "colab_eval.py"
source = path.read_text(encoding="utf-8")

before = "with torch.inference_mode():"
after = "with torch.no_grad():"

if before in source:
    assert source.count(before) == 1, "変更対象が複数あります。内容を確認してください。"
    updated = source.replace(before, after)
    compile(updated, str(path), "exec")
    path.write_text(updated, encoding="utf-8")
    print("評価処理を torch.no_grad() に変更しました。")
elif after in source:
    print("すでに変更済みです。")
else:
    raise RuntimeError("変更対象が見つかりません。colab_eval.pyを確認してください。")

In [ ]:
EVAL_SECONDS = 30.0
EVAL_EPISODES = 5
EVAL_SEED = 1000
MAX_SPEED_ERROR = 0.15

# 再実行するたびに、新しい評価フォルダを使う
EVAL_DIR = LOG_DIR / "evaluation" / ("eval-" + uuid4().hex[:12])

cmd = [
    sys.executable, "-X", "faulthandler", "-u",
    str(REPO / "colab_eval.py"),
    "--checkpoint", str(CHECKPOINT),
    "--output", str(EVAL_DIR),
    "--seconds", str(EVAL_SECONDS),
    "--episodes", str(EVAL_EPISODES),
    "--seed", str(EVAL_SEED),
    "--max-speed-error", str(MAX_SPEED_ERROR),
]

print("評価するモデル:", CHECKPOINT)
print("評価を実行しています。終了後にログを表示します。", flush=True)

result = subprocess.run(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)

# 評価フォルダはcolab_eval.pyが作るので、診断ログは別の場所へ保存
diagnostic_log = REPO / f"{EVAL_DIR.name}-diagnostic.log"
diagnostic_log.write_text(result.stdout, encoding="utf-8")

print(result.stdout, flush=True)
print("終了コード:", result.returncode)
print("診断ログ:", diagnostic_log)

# 失敗していたら、結果の読み込みやコピーへ進まない
if result.returncode != 0:
    raise RuntimeError(
        "評価プログラムが停止しました。上に表示された詳細を確認してください。"
    )

metrics = json.loads(
    (EVAL_DIR / "metrics.json").read_text(encoding="utf-8")
)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
shutil.copytree(EVAL_DIR, BACKUP_DIR / "evaluation" / EVAL_DIR.name)
print("評価結果をDriveへ保存しました。")

# 動画を作成  
Viewerウィンドウを開かず、セル12と同じ秒数・先頭seedで1試行を録画します。学習より低速になることがあります。 録画は640×480・25fps。現在のAPIでは、保存先とfpsを start_recording に渡し、stop_recording() で閉じます。 EGL/OpenGLエラーが出たら、学習モデルは残っています。セル12の数値評価とセル15の保存は続けられます。 古い記事の rendered_envs_idx 置換や録画コードを混ぜないでください。

In [ ]:

path = REPO / "colab_eval.py"
source = path.read_text(encoding="utf-8")

before = """                        if recording:
                          env.camera.set_pose("""

after = """                        if recording:
                          x, y, z = env.base_pos[0].detach().cpu().tolist()
                          env.camera.set_pose("""

if before in source:
    updated = source.replace(before, after, 1)
    compile(updated, str(path), "exec")
    path.write_text(updated, encoding="utf-8")
    print("カメラ追従処理を修正しました。")
elif after in source:
    print("すでに修正済みです。")
else:
    raise RuntimeError("変更対象が見つかりません。")


In [ ]:
VIDEO_DIR = EVAL_DIR / ("video-" + uuid4().hex[:12])
render_env = dict(os.environ, PYOPENGL_PLATFORM="egl")
subprocess.run([sys.executable, "-u", str(REPO / "colab_eval.py"),
                "--checkpoint", str(CHECKPOINT), "--output", str(VIDEO_DIR),
                "--seconds", str(EVAL_SECONDS), "--episodes", "1", "--seed", str(EVAL_SEED),
                "--max-speed-error", str(MAX_SPEED_ERROR), "--video"], cwd=REPO, env=render_env, check=True)
VIDEO_PATH = VIDEO_DIR / "go2.mp4"
assert VIDEO_PATH.is_file() and VIDEO_PATH.stat().st_size > 0
shutil.copytree(VIDEO_DIR, BACKUP_DIR / "evaluation" / EVAL_DIR.name / VIDEO_DIR.name)
print("動画:", VIDEO_PATH)


# 動画を見る  
足が交互に動くことだけでなく、前進しているか、胴体が安定しているか、頻繁に開始地点へ戻っていないかを確認します。 開始地点に突然戻る場合は、姿勢崩れによるリセットの可能性があります。

In [ ]:
from IPython.display import Video, display
display(Video(str(VIDEO_PATH), embed=True, width=640))


In [ ]:
import csv
import statistics

with (VIDEO_DIR / "trajectory.csv").open(
    newline="", encoding="utf-8"
) as f:
    rows = list(csv.DictReader(f))

# 切替直後の加速を避け、各区間の最後の2秒を比較する
for start, end, target in [
    (8, 10, 0.2),
    (18, 20, 0.5),
    (28, 30, 0.8),
]:
    speeds = [
        float(row["forward_speed_m_s"])
        for row in rows
        if start < float(row["time_s"]) <= end
        and row["terminated"] == "False"
        and row["forward_speed_m_s"]
    ]

    if speeds:
        print(
            f"目標 {target:.1f} m/s → "
            f"平均実測 {statistics.mean(speeds):.3f} m/s"
        )
    else:
        print(f"目標 {target:.1f} m/s → 有効な測定値なし")

print(
    "リセット回数:",
    sum(row["terminated"] == "True" for row in rows),
)

# 評価結果・動画もDriveへ保存  
チェックポイントは既に保存されていますが、後から作成した評価結果と動画もコピーします。 動画作成に失敗した場合でも、このセルで他の結果を保存できます。 保存成功後、Driveの PhysicalAI/Go2/実験名 を開いて確認します。

In [ ]:
shutil.copytree(LOG_DIR, BACKUP_DIR, dirs_exist_ok=True)
print("保存先:", BACKUP_DIR)
for path in sorted(BACKUP_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(BACKUP_DIR), path.stat().st_size, "bytes")